# TABA H9ETS — FEMTO/PRONOSTIA Green AI: Full Pipeline

Reproduces every experiment behind the final report: ingestion + validation gate, the 3 data-centric strategies (reference / half-rate / top-10-features) x 4 models x 3 conditions, and the model-compression strategy (pruning + quantization) combined with top-10-features.

Upload the FEMTO/PRONOSTIA `Learning_set` (as a zip containing `Bearing1_1/acc_*.csv` etc.) when prompted.

In [ ]:
!pip -q install codecarbon

import io, pickle, time, zipfile, pathlib
import numpy as np
import pandas as pd
from scipy import stats
from scipy.fft import rfft, rfftfreq
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from codecarbon import OfflineEmissionsTracker

RANDOM_STATE = 42
CRITICAL_FRAC = 0.20
FS_NATIVE = 25600
SAMPLE_INTERVAL_S = 10
FREQ_BANDS_HZ = [(0, 1000), (1000, 5000), (5000, FS_NATIVE // 2)]
np.random.seed(RANDOM_STATE)

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next((n for n in uploaded if n.lower().endswith(".zip")), None)
if zip_name is None:
    raise ValueError("Upload a .zip containing Learning_set (or Bearing*/acc_*.csv directly).")

DATA_DIR = pathlib.Path("/content/femto")
import shutil
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
    z.extractall(DATA_DIR)

csv_files = sorted(DATA_DIR.rglob("acc_*.csv"))
print("Acceleration CSV files found:", len(csv_files))
if len(csv_files) == 0:
    raise ValueError("No acc_*.csv files found — confirm the zip contains Bearing*/acc_*.csv.")

## Feature extraction (24 features/observation) + validation gate

In [ ]:
FEATURE_NAMES = ["rms", "std", "p2p", "crest", "skew", "kurtosis", "energy",
                 "spec_centroid", "spec_entropy", "band_low", "band_mid", "band_high"]

def read_measurement(path):
    df = pd.read_csv(path, header=None)
    df = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    if df.shape[1] < 2:
        raise ValueError(f"File with fewer than two columns: {path}")
    return df.iloc[:, -2].to_numpy(dtype=float), df.iloc[:, -1].to_numpy(dtype=float)

def band_energy_ratios(signal, fs, bands):
    spectrum = np.abs(rfft(signal))
    freqs = rfftfreq(len(signal), d=1.0 / fs)
    power = spectrum ** 2
    total = power.sum()
    if total <= 0:
        return [0.0 for _ in bands]
    return [float(power[(freqs >= lo) & (freqs < hi)].sum() / total) for lo, hi in bands]

def extract_channel_features(signal, fs):
    signal = signal.astype(float)
    rms = float(np.sqrt(np.mean(signal ** 2)))
    std = float(np.std(signal))
    p2p = float(np.ptp(signal))
    crest = float(np.max(np.abs(signal)) / rms) if rms > 0 else 0.0
    skew = float(stats.skew(signal))
    kurt = float(stats.kurtosis(signal))
    energy = float(np.sum(signal ** 2))
    spectrum = np.abs(rfft(signal))
    freqs = rfftfreq(len(signal), d=1.0 / fs)
    power = spectrum ** 2
    total_power = power.sum()
    if total_power > 0:
        centroid = float(np.sum(freqs * power) / total_power)
        p_norm = power / total_power
        p_norm = p_norm[p_norm > 0]
        spec_entropy = float(-np.sum(p_norm * np.log2(p_norm)) / np.log2(len(p_norm)))
    else:
        centroid, spec_entropy = 0.0, 0.0
    return [rms, std, p2p, crest, skew, kurt, energy, centroid, spec_entropy] + band_energy_ratios(signal, fs, FREQ_BANDS_HZ)

def build_feature_table(files_list, fs):
    by_bearing = {}
    for path in files_list:
        by_bearing.setdefault(path.parent.name, []).append(path)
    for b in by_bearing:
        by_bearing[b] = sorted(by_bearing[b])
    rows = []
    for bearing, paths in by_bearing.items():
        n = len(paths)
        for i, path in enumerate(paths):
            h, v = read_measurement(path)
            h_feats = extract_channel_features(h, fs)
            v_feats = extract_channel_features(v, fs)
            rul_min = (n - 1 - i) * SAMPLE_INTERVAL_S / 60
            row = {"bearing": bearing, "obs_index": i, "n_obs_bearing": n, "rul_min": rul_min}
            row.update({f"h_{name}": val for name, val in zip(FEATURE_NAMES, h_feats)})
            row.update({f"v_{name}": val for name, val in zip(FEATURE_NAMES, v_feats)})
            rows.append(row)
    return pd.DataFrame(rows)

t0 = time.time()
features = build_feature_table(csv_files, FS_NATIVE)
print(f"Feature extraction: {time.time() - t0:.1f}s")
feature_columns = [c for c in features.columns if c.startswith("h_") or c.startswith("v_")]
features["rul_frac"] = features["rul_min"] / features.groupby("bearing")["rul_min"].transform("max")
print("Rows:", len(features), "| Bearings:", features["bearing"].nunique(), "| Features:", len(feature_columns))

In [ ]:
# Validation gate — must pass before any modelling
errors = []
if len(feature_columns) != 24:
    errors.append(f"Expected 24 feature columns, found {len(feature_columns)}.")
if features[feature_columns].isna().sum().sum() > 0:
    errors.append("NaNs present in feature table.")
nunique = features[feature_columns].nunique()
near_constant = nunique[nunique <= max(1, int(0.01 * len(features)))]
if len(near_constant) > 0:
    errors.append(f"Near-constant features: {list(near_constant.index)}")
h_cols = [c for c in feature_columns if c.startswith("h_")]
v_cols = [c.replace("h_", "v_", 1) for c in h_cols]
identical_frac = (features[h_cols].to_numpy() == features[v_cols].to_numpy()).mean()
if identical_frac > 0.5:
    errors.append(f"h/v channels identical in {identical_frac:.0%} of cells.")
if errors:
    raise ValueError("VALIDATION GATE FAILED:\n- " + "\n- ".join(errors))
print("Validation gate passed.")

## Table 1 / Table 2 — reference, half-rate, top-10-features x 4 models x 3 conditions

In [ ]:
CONDITIONS = {1: ("Bearing1_1", "Bearing1_2"), 2: ("Bearing2_1", "Bearing2_2"), 3: ("Bearing3_1", "Bearing3_2")}

def measure(fn):
    tracker = OfflineEmissionsTracker(log_level="error", save_to_file=False, country_iso_code="IRL")
    tracker.start()
    t0 = time.time()
    result = fn()
    elapsed = time.time() - t0
    tracker.stop()
    energy_kwh = tracker.final_emissions_data.energy_consumed
    co2_g = tracker.final_emissions_data.emissions * 1000
    return result, elapsed, energy_kwh, co2_g

def lead_time_minutes(test_df_sorted, y_pred_frac):
    truly_critical = test_df_sorted["rul_frac"].to_numpy() <= CRITICAL_FRAC
    if not truly_critical.any():
        return np.nan
    correct_alarm = truly_critical & (y_pred_frac <= CRITICAL_FRAC)
    if not correct_alarm.any():
        return 0.0
    return float(test_df_sorted["rul_min"].to_numpy()[np.argmax(correct_alarm)])

def top_k_features(X_train, y_train, columns, k=10):
    rf = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_train, y_train)
    order = np.argsort(rf.feature_importances_)[::-1]
    return [columns[i] for i in order[:k]]

def rebuild_half_rate(bearing_name, obs_indices):
    bpaths = sorted((DATA_DIR).rglob("acc_*.csv"))
    bpaths = [p for p in bpaths if p.parent.name == bearing_name]
    rows = []
    for i in obs_indices:
        h, v = read_measurement(bpaths[i])
        fs_ds = FS_NATIVE // 2
        h_feats = extract_channel_features(h[::2], fs_ds)
        v_feats = extract_channel_features(v[::2], fs_ds)
        row = {f"h_{n}": val for n, val in zip(FEATURE_NAMES, h_feats)}
        row.update({f"v_{n}": val for n, val in zip(FEATURE_NAMES, v_feats)})
        rows.append(row)
    return pd.DataFrame(rows)[feature_columns]

def get_model(name):
    if name == "Ridge": return Ridge(alpha=1.0, random_state=RANDOM_STATE)
    if name == "HistGradientBoosting": return HistGradientBoostingRegressor(random_state=RANDOM_STATE)
    if name == "RandomForest": return RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
    if name == "MLP": return MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=2000, early_stopping=True, random_state=RANDOM_STATE)
    raise ValueError(name)

MODEL_NAMES = ["Ridge", "HistGradientBoosting", "RandomForest", "MLP"]
STRATEGIES = ["reference", "half-rate", "top-10-features"]
results = []

for cond, (train_b, test_b) in CONDITIONS.items():
    train_df = features[features["bearing"] == train_b].sort_values("obs_index").reset_index(drop=True)
    test_df = features[features["bearing"] == test_b].sort_values("obs_index").reset_index(drop=True)
    for strategy in STRATEGIES:
        if strategy == "reference":
            X_train_raw, X_test_raw, cols = train_df[feature_columns].to_numpy(), test_df[feature_columns].to_numpy(), feature_columns
        elif strategy == "half-rate":
            X_train_raw = rebuild_half_rate(train_b, train_df["obs_index"]).to_numpy()
            X_test_raw = rebuild_half_rate(test_b, test_df["obs_index"]).to_numpy()
            cols = feature_columns
        else:
            scaler_tmp = StandardScaler().fit(train_df[feature_columns])
            cols = top_k_features(scaler_tmp.transform(train_df[feature_columns]), train_df["rul_frac"].to_numpy(), feature_columns, k=10)
            X_train_raw, X_test_raw = train_df[cols].to_numpy(), test_df[cols].to_numpy()
        scaler = StandardScaler().fit(X_train_raw)
        X_train, X_test = scaler.transform(X_train_raw), scaler.transform(X_test_raw)
        y_train, y_test = train_df["rul_frac"].to_numpy(), test_df["rul_frac"].to_numpy()
        for model_name in MODEL_NAMES:
            model = get_model(model_name)
            _, train_time, train_energy_kwh, train_co2_g = measure(lambda: model.fit(X_train, y_train))
            y_pred, infer_time, infer_energy_kwh, infer_co2_g = measure(lambda: model.predict(X_test))
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            truly_critical = y_test <= CRITICAL_FRAC
            recall = float(((y_pred <= CRITICAL_FRAC) & truly_critical).sum() / truly_critical.sum()) if truly_critical.sum() > 0 else np.nan
            lt = lead_time_minutes(test_df, y_pred)
            size_kb = len(pickle.dumps(model)) / 1024
            results.append({"condition": cond, "strategy": strategy, "model": model_name, "n_features": len(cols),
                            "mae_frac": mae, "r2": r2, "critical_recall": recall, "lead_time_min": lt,
                            "train_energy_kwh": train_energy_kwh, "co2_g": train_co2_g + infer_co2_g, "model_size_kb": size_kb})
            print(f"cond{cond} {strategy:16s} {model_name:22s} MAE={mae:.3f} R2={r2:6.3f} recall={recall:.2f} lead={lt:5.1f}min")

results_df = pd.DataFrame(results)
table1 = results_df.groupby(["strategy", "model"]).agg(mae_frac=("mae_frac","mean"), r2=("r2","mean"),
    critical_recall=("critical_recall","mean"), lead_time_min=("lead_time_min","mean"),
    train_energy_kwh=("train_energy_kwh","mean"), model_size_kb=("model_size_kb","mean")).reset_index()
print("\n=== Table 1 (aggregated) ===")
display(table1)

## Table 3 — model compression (pruning + quantization) combined with top-10-features

In [ ]:
def rf_quantized_predict_and_payload(model, X):
    payload_bytes = 0
    preds = np.zeros((len(model.estimators_), X.shape[0]))
    for i, tree in enumerate(model.estimators_):
        t = tree.tree_
        threshold_q = t.threshold.astype(np.float32)
        value_q = t.value.astype(np.float32)
        payload_bytes += threshold_q.nbytes + value_q.nbytes
        node = np.zeros(X.shape[0], dtype=int)
        for _ in range(t.max_depth + 1):
            is_leaf = t.children_left[node] == t.children_right[node]
            feat = t.feature[node]
            go_left = X[np.arange(X.shape[0]), np.clip(feat, 0, X.shape[1] - 1)] <= threshold_q[node]
            next_node = np.where(go_left, t.children_left[node], t.children_right[node])
            node = np.where(is_leaf, node, next_node)
        preds[i] = value_q[node, 0, 0]
    return preds.mean(axis=0), payload_bytes / 1024

COMP_RESULTS = []
for cond, (train_b, test_b) in CONDITIONS.items():
    train_df = features[features["bearing"] == train_b].sort_values("obs_index").reset_index(drop=True)
    test_df = features[features["bearing"] == test_b].sort_values("obs_index").reset_index(drop=True)
    scaler_tmp = StandardScaler().fit(train_df[feature_columns])
    top10 = top_k_features(scaler_tmp.transform(train_df[feature_columns]), train_df["rul_frac"].to_numpy(), feature_columns, k=10)
    scaler = StandardScaler().fit(train_df[top10])
    X_train, X_test = scaler.transform(train_df[top10]), scaler.transform(test_df[top10])
    y_train, y_test = train_df["rul_frac"].to_numpy(), test_df["rul_frac"].to_numpy()

    model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1)
    _, train_time, train_energy_kwh, train_co2_g = measure(lambda: model.fit(X_train, y_train))
    (y_pred, payload_kb), infer_time, infer_energy_kwh, infer_co2_g = measure(lambda: rf_quantized_predict_and_payload(model, X_test))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    truly_critical = y_test <= CRITICAL_FRAC
    recall = float(((y_pred <= CRITICAL_FRAC) & truly_critical).sum() / truly_critical.sum()) if truly_critical.sum() > 0 else np.nan
    lt = lead_time_minutes(test_df, y_pred)
    COMP_RESULTS.append({"condition": cond, "mae_frac": mae, "r2": r2, "critical_recall": recall,
                          "lead_time_min": lt, "train_energy_kwh": train_energy_kwh, "payload_kb": payload_kb})
    print(f"cond{cond} RF pruned+quantized+top10  MAE={mae:.3f} R2={r2:.3f} recall={recall:.2f} lead={lt:.1f}min size={payload_kb:.1f}KB")

table3 = pd.DataFrame(COMP_RESULTS).mean(numeric_only=True)
print("\n=== Table 3 (aggregated: RandomForest pruned+quantized+top-10-features) ===")
display(table3)

## Save outputs

In [ ]:
OUT = pathlib.Path("/content/taba_outputs")
OUT.mkdir(exist_ok=True)
features.to_csv(OUT / "features_reference.csv", index=False)
results_df.to_csv(OUT / "all_results.csv", index=False)
table1.to_csv(OUT / "table1_aggregated.csv", index=False)
pd.DataFrame(COMP_RESULTS).to_csv(OUT / "table3_compression.csv", index=False)
print("Saved to", OUT)
for p in sorted(OUT.iterdir()):
    print(" ", p.name)